# 🚀 FoodSafe-Indic: Free Kaggle GPU Server for Sarvam-1 LLM
This notebook runs **sarvamai/sarvam-1** (2B Indic Open LLM) on a **free Kaggle T4 GPU** and exposes an OpenAI-compatible API via **ngrok**.

### Quick Setup Steps:
1. In the right panel of Kaggle, click **Settings** -> **Accelerator** -> choose **GPU T4 x 2**.
2. Run Cell 1 to install dependencies.
3. Run Cell 2 to start the GPU server and ngrok tunnel.
4. Copy the printed `https://xxxx.ngrok-free.app` URL and paste it into your local project's `.env` file as:
   ```env
   KAGGLE_NGROK_URL=https://xxxx.ngrok-free.app
   ```

In [ ]:
# 1. Install dependencies on Kaggle
!pip install -q fastapi uvicorn pyngrok transformers torch accelerate

In [ ]:
# 2. Load Sarvam-1 LLM & Start Tunnel
import os, sys, time, threading, torch, uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer

# ================================================================
# ⚙️ CONFIGURATION
NGROK_AUTH_TOKEN = "3BlXfdJTmJ4kse7RfApWKdvUzes_7PsogjbcPvkqptd29ztum"
MODEL_ID = "sarvamai/sarvam-1"
PORT = 8000
# ================================================================

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: GPU is NOT enabled! Please turn on GPU in Kaggle settings panel on the right.")

print(f"\n[1/3] Loading tokenizer & model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load on GPU in bfloat16 (~4.5 GB VRAM on T4)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("[2/3] Model successfully loaded on GPU!")

# Define FastAPI application
app = FastAPI(title="FoodSafe-Indic Sarvam-1 Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    messages: List[ChatMessage]
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.2
    model: Optional[str] = MODEL_ID

class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.2

@app.get("/")
@app.get("/health")
def health():
    return {
        "status": "ready",
        "model": MODEL_ID,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "vram_allocated_gb": round(torch.cuda.memory_allocated(0)/(1024**3), 2) if torch.cuda.is_available() else 0.0
    }

def format_prompt(messages: List[ChatMessage]) -> str:
    formatted = ""
    for msg in messages:
        if msg.role == "system":
            formatted += f"System: {msg.content}\n\n"
        elif msg.role == "user":
            formatted += f"Human: {msg.content}\n\n"
        elif msg.role == "assistant":
            formatted += f"Assistant: {msg.content}\n\n"
    formatted += "Assistant: "
    return formatted

def generate_text(prompt: str, max_tokens: int = 512, temperature: float = 0.2) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=max(temperature, 0.01),
            do_sample=temperature > 0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

@app.post("/v1/chat/completions")
def chat_endpoint(req: ChatCompletionRequest):
    prompt = format_prompt(req.messages)
    out = generate_text(prompt, req.max_tokens, req.temperature)
    return {
        "id": "chatcmpl-sarvam1",
        "object": "chat.completion",
        "model": MODEL_ID,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": out}, "finish_reason": "stop"}]
    }

@app.post("/generate")
def generate_endpoint(req: GenerateRequest):
    out = generate_text(req.prompt, req.max_tokens, req.temperature)
    return {"response": out}

# Start web server in background thread (avoids Jupyter asyncio conflict)
print("\n[3/3] Starting web server in background thread...")
server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning"),
    daemon=True
)
server_thread.start()
time.sleep(2)

# Connect ngrok tunnel
try:
    ngrok.kill()
except Exception:
    pass

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(PORT)
public_url = tunnel.public_url

print("\n" + "="*70)
print("🚀 FOODSAFE-INDIC KAGGLE GPU SERVER IS LIVE!")
print("="*70)
print(f"\n🔗 COPY THIS URL AND PASTE INTO YOUR LOCAL .env FILE:\n")
print(f"   KAGGLE_NGROK_URL={public_url}")
print("\n" + "="*70)
print("Keep this cell running while testing queries on your local app.\n")

# Keep cell alive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\nServer stopped.")
    ngrok.kill()
